In [1]:
# This Python 3 environment comes with many helpful analytics libraries installed
# It is defined by the kaggle/python Docker image: https://github.com/kaggle/docker-python
# For example, here's several helpful packages to load

import numpy as np # linear algebra
import pandas as pd # data processing, CSV file I/O (e.g. pd.read_csv)

# Input data files are available in the read-only "../input/" directory
# For example, running this (by clicking run or pressing Shift+Enter) will list all files under the input directory

import os
for dirname, _, filenames in os.walk('/kaggle/input'):
    for filename in filenames:
        print(os.path.join(dirname, filename))

# You can write up to 20GB to the current directory (/kaggle/working/) that gets preserved as output when you create a version using "Save & Run All" 
# You can also write temporary files to /kaggle/temp/, but they won't be saved outside of the current session

/kaggle/input/competitions/playground-series-s6e3/sample_submission.csv
/kaggle/input/competitions/playground-series-s6e3/train.csv
/kaggle/input/competitions/playground-series-s6e3/test.csv
/kaggle/input/datasets/blastchar/telco-customer-churn/WA_Fn-UseC_-Telco-Customer-Churn.csv


In [2]:
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns
import warnings
warnings.filterwarnings('ignore')

from sklearn.preprocessing import LabelEncoder
from sklearn.model_selection import StratifiedKFold
from sklearn.metrics import roc_auc_score

# ── Load Data ─────────────────────────────────────────
train = pd.read_csv('/kaggle/input/competitions/playground-series-s6e3/train.csv')
test  = pd.read_csv('/kaggle/input/competitions/playground-series-s6e3/test.csv')
sub   = pd.read_csv('/kaggle/input/competitions/playground-series-s6e3/sample_submission.csv')

# ── Load Original Dataset ─────────────────────────────
try:
    orig = pd.read_csv('/kaggle/input/datasets/blastchar/telco-customer-churn/WA_Fn-UseC_-Telco-Customer-Churn.csv')
    orig = orig.drop(columns=['customerID'], errors='ignore')
    orig['TotalCharges'] = pd.to_numeric(orig['TotalCharges'], errors='coerce')
    orig['Churn']        = orig['Churn'].map({'Yes': 1, 'No': 0})
    orig['id']           = -1
    print(f'✅ Original dataset loaded: {orig.shape}')
except:
    orig = None
    print('⚠️ Original dataset not found')

# ── Fix Target ─────────────────────────────────────────
if train['Churn'].dtype == object:
    train['Churn'] = train['Churn'].map({'Yes': 1, 'No': 0})
train['Churn'] = train['Churn'].astype(int)

# ── Merge Original ─────────────────────────────────────
if orig is not None:
    orig_cols = [c for c in train.columns if c in orig.columns]
    train     = pd.concat([train, orig[orig_cols]], axis=0, ignore_index=True)
    print(f'✅ Train after merge: {train.shape}')

print(f'Train : {train.shape} | Test : {test.shape}')
print(f'Churn balance:\n{train["Churn"].value_counts(normalize=True).round(3)}')

# ── Constants ──────────────────────────────────────────
TARGET   = 'Churn'
ID_COL   = 'id'
N_SPLITS = 10
SEED     = 42
skf      = StratifiedKFold(n_splits=N_SPLITS, shuffle=True, random_state=SEED)

# ── Feature Engineering ────────────────────────────────
def feature_engineering(df):
    df = df.copy()

    # Encode object cols
    cat_cols = df.select_dtypes(include='object').columns.tolist()
    cat_cols = [c for c in cat_cols if c not in [ID_COL, TARGET]]
    for col in cat_cols:
        df[col] = LabelEncoder().fit_transform(df[col].astype(str))

    # Force numeric
    for col in df.columns:
        if col in [ID_COL, TARGET]: continue
        df[col] = pd.to_numeric(df[col], errors='coerce')

    # Fill NaN
    df = df.fillna(df.median(numeric_only=True))

    # Interactions
    if 'tenure' in df.columns and 'MonthlyCharges' in df.columns:
        df['charge_per_tenure'] = df['MonthlyCharges'] / (df['tenure'] + 1)
        df['total_charge_est']  = df['MonthlyCharges'] * df['tenure']
        df['tenure_x_monthly']  = df['tenure'] * df['MonthlyCharges']

    if 'TotalCharges' in df.columns and 'MonthlyCharges' in df.columns:
        df['charge_ratio']  = df['TotalCharges'] / (df['MonthlyCharges'] + 1)
        df['charges_diff']  = df['TotalCharges'] - df['MonthlyCharges'] * df['tenure'] \
                              if 'tenure' in df.columns else df['TotalCharges'] - df['MonthlyCharges']

    if 'tenure' in df.columns:
        df['tenure_squared']   = df['tenure'] ** 2
        df['tenure_log']       = np.log1p(df['tenure'])
        df['is_new_customer']  = (df['tenure'] <= 3).astype(int)
        df['is_long_customer'] = (df['tenure'] >= 48).astype(int)

    if 'MonthlyCharges' in df.columns:
        df['monthly_log']   = np.log1p(df['MonthlyCharges'])
        df['is_high_payer'] = (df['MonthlyCharges'] > 65).astype(int)

    if 'TotalCharges' in df.columns:
        df['total_log'] = np.log1p(df['TotalCharges'])

    # Service count
    service_cols = [c for c in df.columns if c in [
        'PhoneService','MultipleLines','InternetService',
        'OnlineSecurity','OnlineBackup','DeviceProtection',
        'TechSupport','StreamingTV','StreamingMovies'
    ]]
    if service_cols:
        df['num_services'] = df[service_cols].sum(axis=1)

    # Aggregate stats
    num_cols = df.select_dtypes(include=np.number).columns.tolist()
    num_cols = [c for c in num_cols if c not in [ID_COL, TARGET]]
    df['num_mean'] = df[num_cols].mean(axis=1)
    df['num_std']  = df[num_cols].std(axis=1)
    df['num_max']  = df[num_cols].max(axis=1)
    df['num_min']  = df[num_cols].min(axis=1)

    return df

train_fe = feature_engineering(train)
test_fe  = feature_engineering(test)

FEATURES = [c for c in train_fe.columns if c not in [ID_COL, TARGET]]
X        = train_fe[FEATURES]
y        = train_fe[TARGET]
X_test   = test_fe[FEATURES].reindex(columns=FEATURES, fill_value=0)

print(f'\n✅ Features : {len(FEATURES)}')
print(f'X      : {X.shape}')
print(f'X_test : {X_test.shape}')

✅ Original dataset loaded: (7043, 21)
✅ Train after merge: (601237, 21)
Train : (601237, 21) | Test : (254655, 20)
Churn balance:
Churn
0    0.774
1    0.226
Name: proportion, dtype: float64

✅ Features : 36
X      : (601237, 36)
X_test : (254655, 36)


In [ ]:
from xgboost  import XGBClassifier
from lightgbm import LGBMClassifier
from catboost import CatBoostClassifier
import lightgbm as lgb
import optuna
optuna.logging.set_verbosity(optuna.logging.WARNING)

oof_xgb  = np.zeros(len(X))
oof_lgb  = np.zeros(len(X))
oof_cat  = np.zeros(len(X))
pred_xgb = np.zeros(len(X_test))
pred_lgb = np.zeros(len(X_test))
pred_cat = np.zeros(len(X_test))

# ── Optuna Tune XGBoost (3-fold quick CV) ─────────────
print('🔍 Tuning XGBoost with Optuna (50 trials)...')

def tune_xgb(trial):
    params = dict(
        n_estimators          = trial.suggest_int('n_estimators', 500, 2000),
        learning_rate         = trial.suggest_float('learning_rate', 0.01, 0.1, log=True),
        max_depth             = trial.suggest_int('max_depth', 3, 8),
        min_child_weight      = trial.suggest_int('min_child_weight', 1, 20),
        gamma                 = trial.suggest_float('gamma', 0.0, 1.0),
        subsample             = trial.suggest_float('subsample', 0.5, 1.0),
        colsample_bytree      = trial.suggest_float('colsample_bytree', 0.5, 1.0),
        reg_alpha             = trial.suggest_float('reg_alpha', 0.0, 2.0),
        reg_lambda            = trial.suggest_float('reg_lambda', 0.0, 4.0),
        scale_pos_weight      = (y==0).sum() / (y==1).sum(),
        tree_method           = 'hist',
        device                = 'cuda',
        eval_metric           = 'auc',
        early_stopping_rounds = 50,
        random_state          = SEED,
        n_jobs                = -1
    )
    cv   = StratifiedKFold(n_splits=3, shuffle=True, random_state=SEED)
    aucs = []
    for tr_idx, val_idx in cv.split(X, y):
        m = XGBClassifier(**params)
        m.fit(X.iloc[tr_idx], y.iloc[tr_idx],
              eval_set=[(X.iloc[val_idx], y.iloc[val_idx])],
              verbose=False)
        aucs.append(roc_auc_score(y.iloc[val_idx],
                    m.predict_proba(X.iloc[val_idx])[:, 1]))
    return np.mean(aucs)

study_xgb = optuna.create_study(direction='maximize')
study_xgb.optimize(tune_xgb, n_trials=50, show_progress_bar=True)

best_xgb_params = study_xgb.best_params
best_xgb_params.update(dict(
    scale_pos_weight      = (y==0).sum() / (y==1).sum(),
    tree_method           = 'hist',
    device                = 'cuda',
    eval_metric           = 'auc',
    early_stopping_rounds = 100,
    random_state          = SEED,
    n_jobs                = -1
))
print(f'\n✅ Best XGBoost AUC (3-fold): {study_xgb.best_value:.5f}')
print(f'Best params: {study_xgb.best_params}')

# ── Optuna Tune CatBoost (3-fold quick CV) ────────────
print('\n🔍 Tuning CatBoost with Optuna (30 trials)...')

def tune_cat(trial):
    params = dict(
        iterations         = trial.suggest_int('iterations', 500, 2000),
        learning_rate      = trial.suggest_float('learning_rate', 0.01, 0.1, log=True),
        depth              = trial.suggest_int('depth', 3, 8),
        l2_leaf_reg        = trial.suggest_float('l2_leaf_reg', 1.0, 10.0),
        bootstrap_type     = 'Bernoulli',
        subsample          = trial.suggest_float('subsample', 0.5, 1.0),
        min_data_in_leaf   = trial.suggest_int('min_data_in_leaf', 10, 100),
        auto_class_weights = 'Balanced',
        eval_metric        = 'AUC',
        task_type          = 'GPU',
        random_seed        = SEED,
        verbose            = False
    )
    cv   = StratifiedKFold(n_splits=3, shuffle=True, random_state=SEED)
    aucs = []
    for tr_idx, val_idx in cv.split(X, y):
        m = CatBoostClassifier(**params)
        m.fit(X.iloc[tr_idx], y.iloc[tr_idx],
              eval_set              = (X.iloc[val_idx], y.iloc[val_idx]),
              early_stopping_rounds = 50)
        aucs.append(roc_auc_score(y.iloc[val_idx],
                    m.predict_proba(X.iloc[val_idx])[:, 1]))
    return np.mean(aucs)

study_cat = optuna.create_study(direction='maximize')
study_cat.optimize(tune_cat, n_trials=30, show_progress_bar=True)

best_cat_params = study_cat.best_params
best_cat_params.update(dict(
    bootstrap_type     = 'Bernoulli',
    auto_class_weights = 'Balanced',
    eval_metric        = 'AUC',
    task_type          = 'GPU',
    random_seed        = SEED,
    verbose            = False
))
print(f'\n✅ Best CatBoost AUC (3-fold): {study_cat.best_value:.5f}')
print(f'Best params: {study_cat.best_params}')

🔍 Tuning XGBoost with Optuna (50 trials)...


  0%|          | 0/50 [00:00<?, ?it/s]